# Relative Longitude From Meteorological Time Lags

This notebook estimates a relative east-west axis per `Place_ID` from time-lag correlations in weather time series. The coordinate is relative, band-local, and intended as an XGBoost feature rather than an absolute longitude.


In [ ]:
from pathlib import Path
import itertools
import time

import numpy as np
import pandas as pd
from scipy import linalg


CLEAN_PATHS = {
    "train": Path("data/clean_train.csv"),
    "val": Path("data/clean_val.csv"),
    "test": Path("data/clean_test.csv"),
}
FEATURE_OUTPUT_PATH = Path("data/place_relative_longitude.csv")
DIAGNOSTICS_OUTPUT_PATH = Path("data/relative_longitude_diagnostics.csv")

WRITE_BACK_TO_CLEAN_FILES = True
RELATIVE_LONGITUDE_COL = "relative_longitude"
LATITUDE_COL = "solar_latitude_mean"
TIME_SERIES_COLS = [
    "temperature_2m_above_ground",
    "specific_humidity_2m_above_ground",
]

LATITUDE_BAND_WIDTH = 5
ROLLING_WINDOW_DAYS = 3
MAX_LAG_DAYS = 15
MIN_OVERLAP_DAYS = 25
MIN_CORRELATION = 0.30


def load_longitude_source_data(paths, latitude_col, time_series_cols):
    required_cols = ["Place_ID", "Date", latitude_col, *time_series_cols]
    frames = []

    for split_name, path in paths.items():
        df = pd.read_csv(path, usecols=required_cols)
        df["source_split"] = split_name
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"])

    return df


def preprocess_station_series(df, latitude_col, time_series_cols, latitude_band_width, rolling_window_days):
    agg_spec = {latitude_col: "mean"}
    agg_spec.update({col: "mean" for col in time_series_cols})

    work = (
        df.groupby(["Place_ID", "Date"], as_index=False)
        .agg(agg_spec)
        .sort_values(["Place_ID", "Date"])
    )
    work["latitude_band"] = np.floor(work[latitude_col] / latitude_band_width) * latitude_band_width

    for col in time_series_cols:
        smooth_col = f"{col}_smooth"
        z_col = f"{col}_z"
        work[smooth_col] = work.groupby("Place_ID")[col].transform(
            lambda s: s.rolling(rolling_window_days, min_periods=1).mean()
        )
        station_mean = work.groupby("Place_ID")[smooth_col].transform("mean")
        station_std = work.groupby("Place_ID")[smooth_col].transform("std").replace(0, np.nan)
        work[z_col] = (work[smooth_col] - station_mean) / station_std

    place_meta = (
        work.groupby("Place_ID")
        .agg(
            latitude=(latitude_col, "mean"),
            latitude_band=("latitude_band", lambda s: s.mode().iat[0]),
            n_dates=("Date", "nunique"),
        )
        .reset_index()
    )

    return work, place_meta


def corr_np(x, y, min_overlap_days):
    mask = np.isfinite(x) & np.isfinite(y)
    n_overlap = int(mask.sum())

    if n_overlap < min_overlap_days:
        return np.nan

    x_valid = x[mask]
    y_valid = y[mask]
    x_std = x_valid.std()
    y_std = y_valid.std()

    if x_std == 0 or y_std == 0:
        return np.nan

    return float(((x_valid - x_valid.mean()) * (y_valid - y_valid.mean())).mean() / (x_std * y_std))


def best_pair_lag(arrays, place_i, place_j, max_lag_days, min_overlap_days, min_correlation):
    best_lag = None
    best_corr = -np.inf

    for lag in range(-max_lag_days, max_lag_days + 1):
        lag_corrs = []

        for arr in arrays:
            if lag > 0:
                x = arr[:-lag, place_i]
                y = arr[lag:, place_j]
            elif lag < 0:
                x = arr[-lag:, place_i]
                y = arr[:lag, place_j]
            else:
                x = arr[:, place_i]
                y = arr[:, place_j]

            corr = corr_np(x, y, min_overlap_days)
            if np.isfinite(corr):
                lag_corrs.append(corr)

        if not lag_corrs:
            continue

        combined_corr = float(np.mean(lag_corrs))
        if combined_corr > best_corr:
            best_corr = combined_corr
            best_lag = lag

    if best_lag is None or best_corr < min_correlation:
        return None

    return best_lag, best_corr


def connected_components(n_nodes, equations):
    adjacency = [set() for _ in range(n_nodes)]

    for i, j, _, _ in equations:
        adjacency[i].add(j)
        adjacency[j].add(i)

    seen = set()
    components = []

    for start in range(n_nodes):
        if start in seen:
            continue

        stack = [start]
        seen.add(start)
        component = []

        while stack:
            node = stack.pop()
            component.append(node)

            for neighbor in adjacency[node]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    stack.append(neighbor)

        components.append(sorted(component))

    return components


def solve_component_coordinates(component, equations):
    component_set = set(component)
    component_equations = [
        eq for eq in equations
        if eq[0] in component_set and eq[1] in component_set
    ]

    if len(component) < 2 or not component_equations:
        return None, component_equations

    local_index = {node_index: local_i for local_i, node_index in enumerate(component)}
    A = np.zeros((len(component_equations) + 1, len(component)))
    b = np.zeros(len(component_equations) + 1)

    for row_i, (place_i, place_j, lag, corr) in enumerate(component_equations):
        weight = max(corr, 0.01)
        A[row_i, local_index[place_j]] = weight
        A[row_i, local_index[place_i]] = -weight
        b[row_i] = lag * weight

    A[-1, 0] = 1.0
    b[-1] = 0.0

    coords, *_ = linalg.lstsq(A, b)
    coords = coords - np.nanmean(coords)

    return coords, component_equations


def estimate_relative_longitude(
    work,
    place_meta,
    time_series_cols,
    max_lag_days,
    min_overlap_days,
    min_correlation,
):
    all_dates = pd.date_range(work["Date"].min(), work["Date"].max(), freq="D")
    feature_rows = []
    diagnostic_rows = []

    for latitude_band, band_meta in place_meta.groupby("latitude_band", sort=True):
        places = band_meta["Place_ID"].tolist()
        n_places = len(places)
        band_data = work[work["Place_ID"].isin(places)]

        arrays = []
        for col in time_series_cols:
            pivot = (
                band_data.pivot(index="Date", columns="Place_ID", values=f"{col}_z")
                .reindex(index=all_dates, columns=places)
            )
            arrays.append(pivot.to_numpy(dtype=float))

        equations = []
        for place_i, place_j in itertools.combinations(range(n_places), 2):
            result = best_pair_lag(
                arrays=arrays,
                place_i=place_i,
                place_j=place_j,
                max_lag_days=max_lag_days,
                min_overlap_days=min_overlap_days,
                min_correlation=min_correlation,
            )
            if result is not None:
                lag, corr = result
                equations.append((place_i, place_j, lag, corr))

        components = connected_components(n_places, equations)
        coords_by_index = {}

        for component_id, component in enumerate(components):
            coords, component_equations = solve_component_coordinates(component, equations)

            if coords is None:
                for node_index in component:
                    coords_by_index[node_index] = {
                        RELATIVE_LONGITUDE_COL: np.nan,
                        "longitude_component_id": f"{latitude_band}_{component_id}",
                        "longitude_component_size": len(component),
                        "longitude_component_pairs": len(component_equations),
                        "longitude_band_status": "unresolved_component",
                    }
                continue

            for node_index, coord in zip(component, coords):
                coords_by_index[node_index] = {
                    RELATIVE_LONGITUDE_COL: float(coord),
                    "longitude_component_id": f"{latitude_band}_{component_id}",
                    "longitude_component_size": len(component),
                    "longitude_component_pairs": len(component_equations),
                    "longitude_band_status": "ok",
                }

        for node_index, place_id in enumerate(places):
            feature_rows.append({
                "Place_ID": place_id,
                "latitude_band": latitude_band,
                **coords_by_index[node_index],
            })

        diagnostic_rows.append({
            "latitude_band": latitude_band,
            "n_places": n_places,
            "n_pairs": n_places * (n_places - 1) // 2,
            "n_valid_pairs": len(equations),
            "n_components": len(components),
            "n_resolved_places": sum(
                np.isfinite(coords_by_index[node_index][RELATIVE_LONGITUDE_COL])
                for node_index in range(n_places)
            ),
        })

    relative_longitude = pd.DataFrame(feature_rows)
    diagnostics = pd.DataFrame(diagnostic_rows)

    relative_longitude = relative_longitude.merge(
        place_meta,
        on="Place_ID",
        how="left",
        suffixes=("", "_meta"),
    )

    return relative_longitude, diagnostics


def merge_relative_longitude_into_clean_files(paths, relative_longitude, output_suffix=None):
    feature_cols = ["Place_ID", RELATIVE_LONGITUDE_COL]
    outputs = {}

    for split_name, path in paths.items():
        df = pd.read_csv(path)
        df = df.drop(columns=[RELATIVE_LONGITUDE_COL], errors="ignore")
        merged = df.merge(relative_longitude[feature_cols], on="Place_ID", how="left")

        if output_suffix is None:
            output_path = path
        else:
            output_path = path.with_name(f"{path.stem}{output_suffix}{path.suffix}")

        merged.to_csv(output_path, index=False)
        outputs[split_name] = output_path

    return outputs


start = time.time()
source_df = load_longitude_source_data(CLEAN_PATHS, LATITUDE_COL, TIME_SERIES_COLS)
work_df, place_metadata = preprocess_station_series(
    df=source_df,
    latitude_col=LATITUDE_COL,
    time_series_cols=TIME_SERIES_COLS,
    latitude_band_width=LATITUDE_BAND_WIDTH,
    rolling_window_days=ROLLING_WINDOW_DAYS,
)
relative_longitude_df, relative_longitude_diagnostics = estimate_relative_longitude(
    work=work_df,
    place_meta=place_metadata,
    time_series_cols=TIME_SERIES_COLS,
    max_lag_days=MAX_LAG_DAYS,
    min_overlap_days=MIN_OVERLAP_DAYS,
    min_correlation=MIN_CORRELATION,
)

relative_longitude_df.to_csv(FEATURE_OUTPUT_PATH, index=False)
relative_longitude_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_PATH, index=False)

if WRITE_BACK_TO_CLEAN_FILES:
    updated_clean_paths = merge_relative_longitude_into_clean_files(
        CLEAN_PATHS,
        relative_longitude_df,
        output_suffix=None,
    )
else:
    updated_clean_paths = merge_relative_longitude_into_clean_files(
        CLEAN_PATHS,
        relative_longitude_df,
        output_suffix="_with_relative_longitude",
    )

print(f"Computed relative longitude for {relative_longitude_df[RELATIVE_LONGITUDE_COL].notna().sum()} of {len(relative_longitude_df)} places.")
print(f"Saved {FEATURE_OUTPUT_PATH}")
print(f"Saved {DIAGNOSTICS_OUTPUT_PATH}")
for split_name, output_path in updated_clean_paths.items():
    print(f"Saved {split_name}: {output_path}")
print(f"Runtime: {time.time() - start:.1f} seconds")

relative_longitude_diagnostics
